# Lab 19: API Requests and Caching — Analysis

Run these experiments to see the difference caching makes.

**Before you start:** Make sure your `crypto.py` implementations pass all tests.

**Setup:** Put your CoinGecko Demo API key in the cell below.

In [5]:
import sys
sys.path.insert(0, '../src')

import time
from crypto import get_price, get_prices_batch, CoinCache, get_price_cached

API_KEY = "CG-DCDbYq7mZ4RWc3HVMXA7e2Hj"  # Replace with your CoinGecko Demo key

## Experiment 1: Uncached vs. Cached

Fetch Bitcoin's price 10 times — first without caching, then with caching.
Compare the total time and number of API calls.

In [6]:
# --- Uncached: 10 direct API calls ---
start = time.time()
for i in range(10):
    price = get_price("bitcoin", API_KEY)
uncached_time = time.time() - start

print(f"Uncached: 10 requests in {uncached_time:.2f} seconds")
print(f"Last price: ${price:,.2f}")

Uncached: 10 requests in 0.56 seconds
Last price: $74,141.00


In [7]:
# --- Cached: 10 lookups through cache ---
cache = CoinCache(ttl_seconds=60)

start = time.time()
for i in range(10):
    price = get_price_cached("bitcoin", API_KEY, cache)
cached_time = time.time() - start

print(f"Cached: 10 lookups in {cached_time:.2f} seconds")
print(f"Cache hits: {cache.hits}, Cache misses: {cache.misses}")
print(f"Speedup: {uncached_time / cached_time:.1f}x faster")

Cached: 10 lookups in 0.05 seconds
Cache hits: 9, Cache misses: 1
Speedup: 10.5x faster


### Writeup Questions — Experiment 1

1. How many API calls did the cached version actually make? Why that number?
2. What was the approximate speedup? Why is the difference so large?
3. Is there any downside to this speedup? What are you giving up?

*Your answers here:*

1. It made 1 API call. The first request was a cache miss, so it called the API. After that, the remaining 9 requests were cache hits and used the stored value instead of calling the API again.
2. The speedup was about 10.5x faster. The difference is large because API calls take time over the internet, while cached values are returned instantly from memory.
3. The downside is that the data might be outdated. Since the price is cached, it may not reflect the most current market value.

## Experiment 2: TTL Exploration

Try three different TTL values. For each one, simulate a pattern
of lookups spaced 2 seconds apart and observe the hit rate.

In [10]:
ttl_values = [1, 5, 30]

for ttl in ttl_values:
    cache = CoinCache(ttl_seconds=ttl)
    
    for i in range(6):
        price = get_price_cached("bitcoin", API_KEY, cache)
        if i < 5:  # Don't sleep after last lookup
            time.sleep(2)
    
    total = cache.hits + cache.misses
    hit_rate = cache.hits / total * 100
    print(f"TTL={ttl:2d}s: {cache.hits} hits, {cache.misses} misses, hit rate={hit_rate:.0f}%")

TTL= 1s: 0 hits, 6 misses, hit rate=0%
TTL= 5s: 4 hits, 2 misses, hit rate=67%
TTL=30s: 5 hits, 1 misses, hit rate=83%


### Writeup Questions — Experiment 2

1. With TTL=1 second and lookups every 2 seconds, what hit rate do you expect? Did the results match?
2. If you were building a portfolio tracker that updates every time you open the app, what TTL would you choose? Explain your reasoning.
3. Is there a scenario where you'd want a TTL of 0 (no caching at all)? What about a TTL of infinity (cache forever)?

*Your answers here:*

1. With TTL = 1 second and checking every 2 seconds, I expect 0% hit rate because the cache expires before each new request. Yes, the results matched (0 hits, 6 misses).
2. I would choose a TTL around 5–30 seconds. This keeps the data fairly fresh while still reducing API calls and improving speed.
3. TTL = 0 would be used when you need completely up-to-date data every time (no caching).
TTL = infinity would be used when the data never changes or doesn’t need to be updated often.

## Experiment 3: Batch Efficiency

Compare fetching 5 coins one at a time vs. in a single batch request.

In [11]:
coins = ["bitcoin", "ethereum", "solana", "cardano", "dogecoin"]

# --- Individual calls ---
start = time.time()
individual_prices = {}
for coin in coins:
    individual_prices[coin] = get_price(coin, API_KEY)
individual_time = time.time() - start

print(f"Individual: {len(coins)} calls in {individual_time:.2f} seconds")

# --- Batch call ---
start = time.time()
batch_prices = get_prices_batch(coins, API_KEY)
batch_time = time.time() - start

print(f"Batch: 1 call in {batch_time:.2f} seconds")
print(f"Speedup: {individual_time / batch_time:.1f}x faster")

# Show the prices
print("\nPrices:")
for coin, price in batch_prices.items():
    print(f"  {coin}: ${price:,.2f}")

Individual: 5 calls in 0.55 seconds
Batch: 1 call in 0.13 seconds
Speedup: 4.3x faster

Prices:
  bitcoin: $73,944.00
  ethereum: $2,315.69
  solana: $83.47
  cardano: $0.24
  dogecoin: $0.09


### Writeup Questions — Experiment 3

1. How much faster was the batch call? Where does that time saving come from?
2. Batching and caching are both ways to reduce API calls. When would you use one vs. the other? Can you use both?

*Your answers here:*

1. The batch call was about 4.3x faster. The time saving comes from making one API request instead of five, which reduces network delay and overhead.
2. Batching is better when you need a bunch of different data at once. Caching is better when you’re asking for the same thing over and over. You can definitely use both together to make things even faster and more efficient.